# Language-Model Fine-Tuning — DIMER E2E Tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/language-model-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/language-model-pipeline/blob/main/tutorials/language_model_finetuning_colab.ipynb)

**Profile:** `E2E`  
**Notebook specification:** DIMER Notebook Specification `1.0`

This notebook demonstrates the user-facing **language-model supervised fine-tuning** capability: resolve an approved causal language model from the repository registry, normalize conversational training data through the repository data contract, record a deterministic pre-adaptation baseline, train a QLoRA/PEFT adapter, compare base and adapted behavior, run new-prompt inference, export the deployable adapter contract, and prove that the serialized adapter reconstructs in a fresh model load.

**What the upstream model supplies.** The pinned upstream checkpoint supplies the pretrained/instruction-tuned causal language model and tokenizer/chat template. **What this repository adds.** `lmpipeline` supplies the approved-model registry, data normalization and validation contracts, assistant-only masking, QLoRA setup, tutorial training operation, generation path, artifact packaging, provenance, and artifact-consumption checks. The notebook calls those repository APIs; it does not reimplement the model/training path in cells.

**Adaptation semantics.** Gradient training occurs only for PEFT LoRA parameters. The base model remains frozen and is held in 4-bit NF4 for QLoRA. This is fine-tuning, not in-context learning and not pure inference.

**By the end of this notebook you will be able to:**
- resolve a canonical model key to an immutable upstream revision;
- prepare either the pinned public tutorial sample or your own JSONL/ZIP dataset through the canonical dataset path;
- inspect assistant-only supervision and deterministic baseline behavior;
- train and evaluate a QLoRA adapter without treating optimization loss as task-quality evidence;
- run inference on new prompts and export machine-readable results;
- export a manifested PEFT adapter plus tokenizer/provenance and verify a fresh reconstruction.

**This notebook does not demonstrate:** full-parameter fine-tuning, preference optimization, multimodal models, embeddings, classification, retrieval, a calibrated quality benchmark, production safety validation, or deployment fitness. A successful run proves the demonstrated workflow executes for the selected model/data path; it does not prove the resulting adapter is accurate, safe, or suitable for production.

Repository references: [README](../README.md) · [DIMER profile card](../DIMER_PROFILE_CARD.md) · [dataset contract](../DATASET_SPEC.md) · [training contract](../TRAINING_SPEC.md) · [artifact contract](../ARTIFACT_SPEC.md) · [security](../SECURITY.md).

## Prerequisites and data handling

- **Runtime:** Google Colab or Jupyter with a CUDA GPU. The default path is intended for a Colab T4-class GPU; CPU-only execution is not supported for QLoRA.
- **Network:** the default path downloads the pinned repository revision, pinned Python packages, a pinned public sample dataset, and a pinned Hugging Face model revision.
- **Knowledge:** basic Python and a high-level understanding of causal language models.
- **BYOD schema:** choose `Bring Your Own Dataset` to upload `train.jsonl`, optionally `validation.jsonl` (or `val.jsonl`) and `test.jsonl`; alternatively upload one ZIP containing those files at its content root. Each JSONL file must use one supported schema family: `messages`, `prompt`/`completion`, or `instruction`/`input`/`output`.

**BYOD privacy boundary.** Uploaded dataset bytes remain in the notebook runtime and are not sent by this notebook to the model host or another external service. The notebook does make network requests for package/model/sample acquisition. Do not upload confidential, restricted, personal, or sensitive data to a hosted notebook environment unless you are authorized to place that data there.

Run the notebook from top to bottom. Optional upload dialogs are gated by the data-source control and do not interrupt the default sample path.

## 1. Install and verify the reproducible runtime

The tutorial uses an immutable revision of this repository and exact user-space package versions. PyTorch is accelerator-coupled: rather than replace Colab's CUDA wheel, the notebook verifies that its semantic version matches the lock and records the full Torch/CUDA build. If the runtime does not match, execution fails before model or dataset work begins.

In [ ]:
%pip -q install transformers==5.16.1 tokenizers==0.23.2 huggingface-hub==1.30.0 peft==0.20.0 accelerate==1.14.0 bitsandbytes==0.49.0 safetensors==0.8.0 datasets==4.8.5 pandas==2.3.3 PyYAML==6.0.3 Jinja2==3.1.6
%pip -q install --no-deps git+https://github.com/kurtvalcorza/language-model-pipeline.git@592ab4da1118929ad67f576cfa4dbf2b33c8463d

In [ ]:
import gc
import json
import shutil
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset

from lmpipeline.datasets.normalize import load_examples
from lmpipeline.datasets.resolver import resolve_dataset
from lmpipeline.tutorial_api import (
    TutorialTrainingConfig,
    assert_no_split_leakage,
    assert_tutorial_runtime,
    attach_adapter,
    build_masked_example,
    canonical_dataset_digest,
    consume_adapter_archive,
    export_adapter_bundle,
    generate_reply,
    load_adapter_for_inference,
    load_base_model,
    load_tokenizer,
    normalize_records,
    resolve_tutorial_model,
    seed_everything,
    sha256_file,
    tokenize_splits,
    train_adapter,
    trainable_parameter_summary,
    zip_directory,
)

RUNTIME = assert_tutorial_runtime()
if not torch.cuda.is_available():
    raise RuntimeError("QLoRA requires a CUDA GPU. In Colab choose Runtime > Change runtime type > T4 GPU.")
print(json.dumps(RUNTIME, indent=2))

## 2. Resolve the model and seed stochastic operations

`BASE_MODEL_KEY` is resolved by `ModelRegistry` inside the repository API; the notebook contains no second model registry. The selected entry must be enabled, must support QLoRA, must use an immutable 40-character revision, and must accept the requested sequence length. The default is a user-facing Qwen3 model.

The seed is applied **before tokenizer/model/adapter construction and training** to Python, NumPy when present, Torch CPU, and all visible CUDA devices. The printed determinism record also names residual GPU/quantized-kernel variability; reproducibility here does not mean bitwise equality across different hardware.

In [ ]:
BASE_MODEL_KEY = "qwen3-1.7b" # @param {type:"string"}
MAX_SEQUENCE_LENGTH = 512 # @param {type:"integer"}
EPOCHS = 1 # @param {type:"integer"}
LEARNING_RATE = 0.0002 # @param {type:"number"}
LORA_RANK = 8 # @param {type:"integer"}
LORA_ALPHA = 16 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}

DETERMINISM = seed_everything(SEED)
entry = resolve_tutorial_model(
    BASE_MODEL_KEY,
    method="qlora",
    max_sequence_length=MAX_SEQUENCE_LENGTH,
)
print(json.dumps(DETERMINISM, indent=2))
print({
    "modelKey": entry.key,
    "modelId": entry.model_id,
    "revision": entry.revision,
    "license": entry.license,
    "maxSequenceLength": entry.max_sequence_length,
    "servingProfile": entry.serving_profile,
})

## 3. Load the default sample or your own dataset

The **default sample** is `jpaulpoliquit/ph-sft-ai-authored-v1` at immutable dataset revision `8333699c6cc7296cc69cefc09def010851ded919` (Apache-2.0). It is public tutorial/sanity data, not benchmark evidence. The notebook deterministically considers records by canonical fingerprint and keeps up to `SAMPLE_LIMIT` examples that fit the selected model's token window; every skipped over-length record is counted and reported.

For BYOD, uploaded files go through the repository's `resolve_dataset` and `load_examples` code path. Supplied validation/test splits are preserved. If validation is absent, the repository tutorial API derives an order-independent validation split from content hashes and `SEED`. Exact overlap across splits is fatal. No row is silently truncated.

In [ ]:
DATA_SOURCE = "Sample: Filipino SFT" # @param ["Sample: Filipino SFT","Bring Your Own Dataset"]
SAMPLE_LIMIT = 120 # @param {type:"integer"}
WORK_DIR = Path("/content/language-model-tutorial")
shutil.rmtree(WORK_DIR, ignore_errors=True)
WORK_DIR.mkdir(parents=True)

tokenizer = load_tokenizer(entry)

if DATA_SOURCE == "Sample: Filipino SFT":
    dataset_id = "jpaulpoliquit/ph-sft-ai-authored-v1"
    dataset_revision = "8333699c6cc7296cc69cefc09def010851ded919"
    source_rows = [dict(row) for row in load_dataset(dataset_id, revision=dataset_revision, split="train")]
    normalized = sorted(normalize_records(source_rows), key=lambda item: item.fingerprint())
    selected = []
    skipped_over_length = 0
    for item in normalized:
        if len(selected) >= SAMPLE_LIMIT:
            break
        try:
            build_masked_example(tokenizer, item, max_sequence_length=MAX_SEQUENCE_LENGTH)
        except Exception as exc:
            if "renders to" in str(exc) and "tokens" in str(exc):
                skipped_over_length += 1
                continue
            raise
        selected.append(item)
    RAW_SPLITS = {"train": selected}
    DATASET_PROVENANCE = {
        "source": dataset_id,
        "revision": dataset_revision,
        "license": "apache-2.0",
        "usage": "tutorial-sanity-not-benchmark",
        "considered": len(normalized),
        "selected": len(selected),
        "skippedOverLengthWhileSelecting": skipped_over_length,
    }
    print(f"sample selection: kept {len(selected)}, skipped {skipped_over_length} over-length records")
else:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No BYOD files were uploaded")
    upload_root = WORK_DIR / "upload"
    upload_root.mkdir()
    for name, payload in uploaded.items():
        (upload_root / Path(name).name).write_bytes(payload)
    resolved = resolve_dataset(upload_root, workdir=WORK_DIR / "resolved")
    RAW_SPLITS = {
        name: load_examples(path, max_examples=500_000)
        for name, path in resolved.splits.items()
    }
    DATASET_PROVENANCE = {
        "source": "BYOD",
        "transport": resolved.source,
        "archive": resolved.archive,
        "usage": "user-provided",
    }

for split_name, examples in RAW_SPLITS.items():
    duplicate_count = len(examples) - len({example.fingerprint() for example in examples})
    if duplicate_count:
        print(f"Warning: {duplicate_count} exact duplicate record(s) inside {split_name}; none removed")
assert_no_split_leakage(RAW_SPLITS)
DATASET_DIGEST = canonical_dataset_digest(RAW_SPLITS)
print({name: len(items) for name, items in RAW_SPLITS.items()})
print("dataset digest:", DATASET_DIGEST)

## 4. Validate token limits and inspect assistant-only loss masking

The model reads the whole rendered conversation, but loss is applied only to assistant-turn tokens. User/system/template tokens receive label `-100` and do not contribute to cross-entropy. The repository API computes assistant spans from the tokenizer's own chat template and refuses a template that is not prefix-stable; it also refuses any example above `MAX_SEQUENCE_LENGTH` rather than truncating it.

The next cell preserves supplied splits, derives validation only if needed, prints the effective split sizes and supervised-token counts, and makes the first training example's supervision visible. Successful output means the data can be rendered and masked for this exact tokenizer/revision; it is not evidence of model quality.

In [ ]:
SPLITS = tokenize_splits(
    RAW_SPLITS,
    tokenizer=tokenizer,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    validation_fraction=0.20,
    seed=SEED,
)
print("effective splits:", SPLITS.counts(), "validation derived:", SPLITS.validation_was_derived)
print("supervised tokens:", {
    "train": sum(item.supervised_token_count for item in SPLITS.train),
    "validation": sum(item.supervised_token_count for item in SPLITS.validation),
    "test": sum(item.supervised_token_count for item in SPLITS.test),
})

first = SPLITS.train[0]
visible = []
inside = False
for token_id, label in zip(first.input_ids, first.labels):
    supervised = label != -100
    if supervised != inside:
        visible.append("⟦" if supervised else "⟧")
        inside = supervised
    visible.append(tokenizer.decode([token_id]))
if inside:
    visible.append("⟧")
print("".join(visible))

## 5. Load the pinned base model and record a deterministic baseline

The repository API loads the canonical model at its immutable revision with `trust_remote_code=False`, NF4 double quantization, an explicit compute dtype, SDPA attention, and the selected GPU. Greedy decoding (`do_sample=False`) is used here so pre/post behavior can be compared without sampling noise.

These prompts are behavioral probes, not a benchmark. They establish what the base model emitted before adaptation.

In [ ]:
loaded = load_base_model(
    entry,
    tokenizer=tokenizer,
    method="qlora",
    device="cuda",
)
BASE_MODEL = loaded.model
PROBE_PROMPTS = [
    "Ipaliwanag sa simpleng Filipino kung ano ang machine learning.",
    "Magbigay ng tatlong paraan para mabawasan ang basura sa opisina.",
]
BASELINE_OUTPUTS = [
    generate_reply(BASE_MODEL, tokenizer, prompt, decoding={"do_sample": False})
    for prompt in PROBE_PROMPTS
]
print("load configuration:", {
    "dtype": loaded.torch_dtype,
    "quantized": loaded.quantized,
    "targetModules": loaded.target_modules,
})
display(pd.DataFrame({"prompt": PROBE_PROMPTS, "base": BASELINE_OUTPUTS}))

## 6. Attach and train the QLoRA adapter

`attach_adapter` and `train_adapter` are repository APIs, so the notebook exercises the supported implementation rather than defining its own trainer. The base remains frozen; only LoRA parameters are trainable. Training reports cross-entropy on supervised assistant tokens and validation/test loss when those splits exist.

Loss and perplexity are **optimization evidence**. They measure how well the model predicts held-out assistant tokens under this tutorial split; they are not accuracy, safety, factuality, preference, or production-fitness metrics.

In [ ]:
MODEL = attach_adapter(loaded, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=0.05)
print("trainable parameters:", trainable_parameter_summary(MODEL))
TRAINING_CONFIG = TutorialTrainingConfig(
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    seed=SEED,
)
METRICS = train_adapter(
    MODEL,
    SPLITS,
    config=TRAINING_CONFIG,
    pad_token_id=tokenizer.pad_token_id,
    device="cuda",
).to_dict()
ADAPTED_OUTPUTS = [
    generate_reply(MODEL, tokenizer, prompt, decoding={"do_sample": False})
    for prompt in PROBE_PROMPTS
]
print(json.dumps(METRICS, indent=2))
display(pd.DataFrame({
    "prompt": PROBE_PROMPTS,
    "base": BASELINE_OUTPUTS,
    "adapted": ADAPTED_OUTPUTS,
}))

## 7. Run inference on new prompts and export machine-readable results

This stage is separate from the validation split. `CUSTOM_PROMPT` is a real user-input path; leave it blank to run the two unseen default prompts only. Outputs are written as JSONL with prompt identity, base/adapted text, decoding settings, model identity, and immutable revision so downstream use is not dependent on notebook display state.

In [ ]:
CUSTOM_PROMPT = "" # @param {type:"string"}
NEW_PROMPTS = [
    "Sumulat ng maikling payo para sa isang estudyanteng nagsisimula sa AI.",
    "Ipaliwanag ang pagkakaiba ng training data at evaluation data sa dalawang pangungusap.",
]
if CUSTOM_PROMPT.strip():
    NEW_PROMPTS.append(CUSTOM_PROMPT.strip())

RESULT_ROWS = []
for prompt in NEW_PROMPTS:
    with MODEL.disable_adapter():
        base_answer = generate_reply(MODEL, tokenizer, prompt, decoding={"do_sample": False})
    adapted_answer = generate_reply(MODEL, tokenizer, prompt, decoding={"do_sample": False})
    RESULT_ROWS.append({
        "prompt": prompt,
        "base": base_answer,
        "adapted": adapted_answer,
        "decoding": {"doSample": False},
        "modelId": entry.model_id,
        "modelRevision": entry.revision,
    })
OUTPUT_JSONL = WORK_DIR / "tutorial_predictions.jsonl"
OUTPUT_JSONL.write_text("
".join(json.dumps(row, ensure_ascii=False) for row in RESULT_ROWS) + "
")
METRICS_JSON = WORK_DIR / "tutorial_metrics.json"
METRICS_JSON.write_text(json.dumps(METRICS, indent=2))
display(pd.DataFrame(RESULT_ROWS)[["prompt", "base", "adapted"]])
print("wrote", OUTPUT_JSONL, "and", METRICS_JSON)

## 8. Export the deployable adapter and verify the serialized boundary

The deployable object is a **PEFT adapter**, not a complete model. It depends on the exact base model and revision recorded in provenance. The bundle contains safe-serialized adapter weights, adapter config, tokenizer/chat-template assets, tutorial metrics, provenance, a model card, and `artifact-manifest.json` with byte counts and SHA-256 for every load-bearing file. The artifact may encode information learned from the training/support data and must be handled under the source data's confidentiality, licensing, retention, and disclosure obligations. Credentials are never written.

The notebook then deletes the training model, reloads the pinned base, attaches the serialized adapter from disk, and generates from a fresh prompt. This proves the artifact **reconstructs and is usable after serialization**. It deliberately does not claim byte-identical or text-identical equivalence with the in-memory training object; generation can diverge across numerically different reconstruction paths.

In [ ]:
ARTIFACT_DIR = WORK_DIR / "dimer-lm-adapter"
ARTIFACT_ZIP = WORK_DIR / "dimer-language-model-adapter.zip"
PROVENANCE = {
    "artifactFormat": "peft_adapter",
    "artifactFormatVersion": 1,
    "baseModel": entry.model_id,
    "baseModelRevision": entry.revision,
    "baseModelLicense": entry.license,
    "modelKey": entry.key,
    "trustRemoteCode": False,
    "dataset": DATASET_PROVENANCE,
    "datasetDigest": DATASET_DIGEST,
    "training": {**TRAINING_CONFIG.to_dict(), "maxSequenceLength": MAX_SEQUENCE_LENGTH},
    "runtime": RUNTIME,
    "determinism": DETERMINISM,
    "tutorial": {"profile": "E2E", "specVersion": "1.0"},
}
export_adapter_bundle(
    MODEL,
    tokenizer,
    destination=ARTIFACT_DIR,
    provenance=PROVENANCE,
    metrics=METRICS,
)
zip_directory(ARTIFACT_DIR, ARTIFACT_ZIP)
ARTIFACT_SHA256 = sha256_file(ARTIFACT_ZIP)
print("artifact SHA-256:", ARTIFACT_SHA256)

del MODEL, BASE_MODEL, loaded
gc.collect()
torch.cuda.empty_cache()

# Consume the serialized directory through the same public artifact API used by the companion.
manifest = json.loads((ARTIFACT_DIR / "artifact-manifest.json").read_text())
if manifest.get("format") != "peft_adapter":
    raise RuntimeError("serialized artifact format is not peft_adapter")
RELOADED_MODEL, RELOADED_TOKENIZER = load_adapter_for_inference(
    entry,
    artifact_root=ARTIFACT_DIR,
)
RELOAD_PROMPT = "Kumusta! Sagutin sa isang maikling pangungusap."
RELOADED_OUTPUT = generate_reply(
    RELOADED_MODEL,
    RELOADED_TOKENIZER,
    RELOAD_PROMPT,
    decoding={"do_sample": False},
)
if not RELOADED_OUTPUT:
    raise RuntimeError("Fresh artifact reconstruction generated no output")
print("fresh reconstruction PASS")
print("fresh prompt:", RELOAD_PROMPT)
print("fresh output:", RELOADED_OUTPUT)
print("This check establishes reconstruction and generation, not text-equivalence with the pre-export object.")

## Interpretation, limits, and next steps

A complete run establishes that the **selected path** resolved an immutable approved model revision, normalized and token-validated the selected data, created assistant-only supervision, recorded a deterministic pre-adaptation baseline, trained a QLoRA adapter through the repository API, produced optimization metrics, generated new-prompt outputs, wrote machine-readable results/provenance, exported a manifested adapter, and reconstructed that serialized adapter against the required base revision.

It does **not** establish benchmark accuracy, factual correctness, safety, fairness, calibration, robustness, domain generalization, production latency/SLA, or deployment fitness. The sample split is tutorial evidence only. For an application decision, use a representative independent test set and task-appropriate human/automatic evaluation that was not used for model selection.

**Residual variability:** the notebook seeds controlled RNGs, but GPU kernels, CUDA/bitsandbytes builds, and hardware can prevent bitwise reproducibility. Greedy generation removes sampling randomness; stochastic decoding should be separately seeded and its parameters recorded.

**Next experiments:** repeat with a representative BYOD train/validation/test split; increase epochs while watching held-out loss for overfitting; compare LoRA rank/alpha settings using validation only; then perform independent task-quality evaluation.

The companion `language_model_artifact_inference_colab.ipynb` consumes an externally supplied adapter ZIP and exercises the downstream trust, reconstruction, user-input, and prediction-export boundary.